# Backtest da Estratégia de Pairs Trading

Este notebook realiza o backtest da estratégia de pairs trading a partir dos sinais gerados no Notebook 05.

O objetivo é simular a estratégia usando os sinais de z-score dinâmico, calcular o resultado de cada operação e avaliar a performance da estratégia.

### Importação das bibliotecas

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.6f}".format)

### Definição dos caminhos dos arquivos

Nesta etapa, definimos os arquivos de entrada e saída.

Usaremos:

- a base de preços com setores;
- a base de sinais gerada pelo Notebook 05.

In [2]:
arquivo_precos = Path("../dados_tratados/dados_economatica_B3_com_setores.parquet")

arquivo_sinais = Path("../dados_tratados/modelo_kalman_sinais.parquet")

arquivo_trades = Path("../dados_tratados/backtest_trades.parquet")

arquivo_metricas = Path("../dados_tratados/backtest_metricas.parquet")

arquivo_pnl_diario = Path("../dados_tratados/backtest_pnl_diario.parquet")

print("Arquivo de preços existe?", arquivo_precos.exists())
print("Arquivo de sinais existe?", arquivo_sinais.exists())

arquivo_trades.parent.mkdir(parents=True, exist_ok=True)

Arquivo de preços existe? True
Arquivo de sinais existe? True


### Carregamento das bases

Nesta etapa, carregamos a base de preços e a base de sinais do modelo.

A base de preços será usada para calcular o retorno dos ativos durante cada operação.

A base de sinais indica quando o modelo enxerga desvio relevante no spread do par.

In [3]:
precos = pd.read_parquet(arquivo_precos)

sinais = pd.read_parquet(arquivo_sinais)

print("Base de preços:", precos.shape)
print("Base de sinais:", sinais.shape)

display(precos.head())
display(sinais.head())

Base de preços: (1374399, 21)
Base de sinais: (67446, 22)


,ticker,data,ativo,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_financeiro,q_titulos,nome,classe,codigo_economatica,isin,id_papel,cnpj,situacao_cvm,setor,subsetor,segmento
0,ABYA3,2010-01-04,ABYA3<XBSP>,4.600000,4.610000,4.570000,4.620000,4.600000,463.000000,"3,991,454.000000","868,600.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
1,ABYA3,2010-01-05,ABYA3<XBSP>,4.580000,4.630000,4.570000,4.630000,4.600000,319.000000,"3,280,211.000000","713,700.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
2,ABYA3,2010-01-06,ABYA3<XBSP>,4.870000,4.570000,4.560000,4.920000,4.800000,"1,715.000000","18,347,711.000000","3,826,000.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
3,ABYA3,2010-01-07,ABYA3<XBSP>,5.170000,4.790000,4.740000,5.180000,5.030000,"2,655.000000","22,244,229.000000","4,420,100.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
4,ABYA3,2010-01-08,ABYA3<XBSP>,5.420000,5.180000,5.180000,5.450000,5.320000,"2,229.000000","21,789,270.000000","4,093,100.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN


,data,alpha_kalman,beta_kalman,spread_kalman,variancia_inovacao,zscore_kalman,data_fim_janela,data_fim_teste,ativo_1,ativo_2,ranking_cointegracao,ranking_distancia,pvalor_coint,distancia,ticker_1,ticker_2,setor_1,setor_2,mesmo_setor,combinacao_setorial,sinal,direcao
0,2012-02-01,0.192486,0.927505,0.017306,13.309462,0.004744,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
1,2012-02-02,0.160653,0.934558,-0.016159,0.030383,-0.092702,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
2,2012-02-03,0.152677,0.937595,0.007120,0.022523,0.047443,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
3,2012-02-06,0.522811,0.844369,0.213666,0.020093,1.507353,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
4,2012-02-07,0.425407,0.868761,-0.073522,0.018405,-0.541934,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO


### Validação das colunas necessárias

Antes de fazer o backtest, verificamos se as bases possuem as colunas necessárias.

Na base de preços, precisamos de:

- data;
- id_papel;
- fechamento_ajustado.

Na base de sinais, precisamos de:

- data;
- data_fim_janela;
- ativo_1;
- ativo_2;
- beta_kalman;
- zscore_kalman;
- sinal.

In [4]:
colunas_precos = [
    "data",
    "id_papel",
    "fechamento_ajustado"
]

colunas_sinais = [
    "data",
    "data_fim_janela",
    "ativo_1",
    "ativo_2",
    "beta_kalman",
    "zscore_kalman",
    "sinal"
]

faltantes_precos = [
    coluna for coluna in colunas_precos
    if coluna not in precos.columns
]

faltantes_sinais = [
    coluna for coluna in colunas_sinais
    if coluna not in sinais.columns
]

if faltantes_precos:
    raise ValueError(f"Colunas faltantes na base de preços: {faltantes_precos}")

if faltantes_sinais:
    raise ValueError(f"Colunas faltantes na base de sinais: {faltantes_sinais}")

print("Todas as colunas obrigatórias estão presentes.")

Todas as colunas obrigatórias estão presentes.


### Preparação das datas e da matriz de preços

Nesta etapa, garantimos que as datas estejam no formato correto.

Também criamos uma matriz de log-preços ajustados, em que:

- as linhas são datas;
- as colunas são ativos identificados por "id_papel";
- os valores são logaritmos dos preços ajustados.

In [5]:
precos = precos.copy()
sinais = sinais.copy()

precos["data"] = pd.to_datetime(precos["data"], errors="coerce")
sinais["data"] = pd.to_datetime(sinais["data"], errors="coerce")
sinais["data_fim_janela"] = pd.to_datetime(sinais["data_fim_janela"], errors="coerce")

precos = precos.dropna(subset=["data", "id_papel", "fechamento_ajustado"])
precos = precos[precos["fechamento_ajustado"] > 0].copy()

matriz_precos = (
    precos
    .pivot_table(
        index="data",
        columns="id_papel",
        values="fechamento_ajustado",
        aggfunc="last"
    )
    .sort_index()
)

matriz_log = np.log(matriz_precos)

sinais = sinais.dropna(
    subset=["data", "data_fim_janela", "ativo_1", "ativo_2", "zscore_kalman", "beta_kalman"]
).copy()

sinais = sinais.sort_values(
    ["data_fim_janela", "ativo_1", "ativo_2", "data"]
).reset_index(drop=True)

print("Matriz de preços:", matriz_precos.shape)
print("Base de sinais após limpeza:", sinais.shape)

Matriz de preços: (4053, 775)
Base de sinais após limpeza: (67446, 22)


### Parâmetros do backtest

Nesta etapa, definimos as regras de entrada, saída, stop e custos.

As regras serão:

- entrada quando |z-score| >= 2;
- saída por convergência quando |z-score| <= 0,5;
- stop por divergência quando |z-score| >= 3;
- stop por tempo quando a posição ficar aberta por mais de 30 pregões.

Também será considerado um custo operacional em bps.

In [6]:
Z_ENTRADA = 2.0

Z_SAIDA = 0.5

Z_STOP = 3.0

STOP_DIAS = 30

CUSTO_BPS = 10

CUSTO_TOTAL = 2 * CUSTO_BPS / 10_000

print("Z de entrada:", Z_ENTRADA)
print("Z de saída:", Z_SAIDA)
print("Z de stop:", Z_STOP)
print("Stop por dias:", STOP_DIAS)
print("Custo total aproximado:", CUSTO_TOTAL)

Z de entrada: 2.0
Z de saída: 0.5
Z de stop: 3.0
Stop por dias: 30
Custo total aproximado: 0.002


### Função de backtest para um par em uma janela

Nesta etapa, criamos uma função que simula as operações de um único par dentro de uma janela.

A função percorre os dias em ordem cronológica e aplica as regras de entrada e saída.

A posição pode ser:

- long no ativo 1 e short no ativo 2;
- short no ativo 1 e long no ativo 2;
- neutra.

O retorno do trade será calculado pelo retorno do spread ajustado pelo beta de entrada.

In [7]:
def backtest_par_janela(dados_par):
    dados_par = dados_par.sort_values("data").copy()
    
    ativo_1 = dados_par["ativo_1"].iloc[0]
    ativo_2 = dados_par["ativo_2"].iloc[0]
    data_fim_janela = dados_par["data_fim_janela"].iloc[0]
    
    trades = []
    
    posicao = None
    data_entrada = None
    z_entrada = None
    beta_entrada = None
    log_1_entrada = None
    log_2_entrada = None
    dias_abertos = 0
    
    for _, linha in dados_par.iterrows():
        data = linha["data"]
        z_atual = linha["zscore_kalman"]
        beta_atual = linha["beta_kalman"]
        
        if data not in matriz_log.index:
            continue
        
        if ativo_1 not in matriz_log.columns or ativo_2 not in matriz_log.columns:
            continue
        
        log_1 = matriz_log.loc[data, ativo_1]
        log_2 = matriz_log.loc[data, ativo_2]
        
        if pd.isna(log_1) or pd.isna(log_2):
            continue
        
        if posicao is None:
            if z_atual >= Z_ENTRADA:
                posicao = "short_ativo_1_long_ativo_2"
                data_entrada = data
                z_entrada = z_atual
                beta_entrada = beta_atual
                log_1_entrada = log_1
                log_2_entrada = log_2
                dias_abertos = 0
                
            elif z_atual <= -Z_ENTRADA:
                posicao = "long_ativo_1_short_ativo_2"
                data_entrada = data
                z_entrada = z_atual
                beta_entrada = beta_atual
                log_1_entrada = log_1
                log_2_entrada = log_2
                dias_abertos = 0
        
        else:
            dias_abertos += 1
            
            fechar = (
                abs(z_atual) <= Z_SAIDA or
                abs(z_atual) >= Z_STOP or
                dias_abertos >= STOP_DIAS
            )
            
            if fechar:
                retorno_1 = log_1 - log_1_entrada
                retorno_2 = log_2 - log_2_entrada
                
                retorno_spread = retorno_1 - beta_entrada * retorno_2
                
                if posicao == "short_ativo_1_long_ativo_2":
                    retorno_spread = -retorno_spread
                
                pnl_bruto = retorno_spread
                pnl_liquido = pnl_bruto - CUSTO_TOTAL
                
                if abs(z_atual) <= Z_SAIDA:
                    razao_saida = "convergencia"
                elif abs(z_atual) >= Z_STOP:
                    razao_saida = "stop_divergencia"
                else:
                    razao_saida = "stop_tempo"
                
                trades.append({
                    "data_fim_janela": data_fim_janela,
                    "ativo_1": ativo_1,
                    "ativo_2": ativo_2,
                    "data_entrada": data_entrada,
                    "data_saida": data,
                    "dias_abertos": dias_abertos,
                    "posicao": posicao,
                    "z_entrada": z_entrada,
                    "z_saida": z_atual,
                    "beta_entrada": beta_entrada,
                    "beta_saida": beta_atual,
                    "pnl_bruto": pnl_bruto,
                    "pnl_liquido": pnl_liquido,
                    "razao_saida": razao_saida
                })
                
                posicao = None
                data_entrada = None
                z_entrada = None
                beta_entrada = None
                log_1_entrada = None
                log_2_entrada = None
                dias_abertos = 0
    
    if posicao is not None:
        ultima_linha = dados_par.iloc[-1]
        data = ultima_linha["data"]
        z_atual = ultima_linha["zscore_kalman"]
        beta_atual = ultima_linha["beta_kalman"]
        
        if data in matriz_log.index and ativo_1 in matriz_log.columns and ativo_2 in matriz_log.columns:
            log_1 = matriz_log.loc[data, ativo_1]
            log_2 = matriz_log.loc[data, ativo_2]
            
            if not pd.isna(log_1) and not pd.isna(log_2):
                retorno_1 = log_1 - log_1_entrada
                retorno_2 = log_2 - log_2_entrada
                
                retorno_spread = retorno_1 - beta_entrada * retorno_2
                
                if posicao == "short_ativo_1_long_ativo_2":
                    retorno_spread = -retorno_spread
                
                trades.append({
                    "data_fim_janela": data_fim_janela,
                    "ativo_1": ativo_1,
                    "ativo_2": ativo_2,
                    "data_entrada": data_entrada,
                    "data_saida": data,
                    "dias_abertos": dias_abertos,
                    "posicao": posicao,
                    "z_entrada": z_entrada,
                    "z_saida": z_atual,
                    "beta_entrada": beta_entrada,
                    "beta_saida": beta_atual,
                    "pnl_bruto": retorno_spread,
                    "pnl_liquido": retorno_spread - CUSTO_TOTAL,
                    "razao_saida": "fim_janela"
                })
    
    return pd.DataFrame(trades)

### Execução do backtest em todos os pares

Nesta etapa, aplicamos a função de backtest para todos os pares e janelas.

Cada grupo é definido por:

- data da janela de formação;
- ativo 1;
- ativo 2.

O resultado será uma base em que cada linha representa um trade concluído.

In [8]:
resultados_trades = []

grupos = sinais.groupby(["data_fim_janela", "ativo_1", "ativo_2"])

for i, (_, dados_par) in enumerate(grupos):
    trades_par = backtest_par_janela(dados_par)
    
    if not trades_par.empty:
        resultados_trades.append(trades_par)
    
    if (i + 1) % 100 == 0:
        print(f"Grupos processados: {i + 1} de {len(grupos)}")

if resultados_trades:
    trades = pd.concat(resultados_trades, ignore_index=True)
else:
    trades = pd.DataFrame()

print("Quantidade de trades:", len(trades))

display(trades.head())

Grupos processados: 100 de 3383
Grupos processados: 200 de 3383
Grupos processados: 300 de 3383
Grupos processados: 400 de 3383
Grupos processados: 500 de 3383
Grupos processados: 600 de 3383
Grupos processados: 700 de 3383
Grupos processados: 800 de 3383
Grupos processados: 900 de 3383
Grupos processados: 1000 de 3383
Grupos processados: 1100 de 3383
Grupos processados: 1200 de 3383
Grupos processados: 1300 de 3383
Grupos processados: 1400 de 3383
Grupos processados: 1500 de 3383
Grupos processados: 1600 de 3383
Grupos processados: 1700 de 3383
Grupos processados: 1800 de 3383
Grupos processados: 1900 de 3383
Grupos processados: 2000 de 3383
Grupos processados: 2100 de 3383
Grupos processados: 2200 de 3383
Grupos processados: 2300 de 3383
Grupos processados: 2400 de 3383
Grupos processados: 2500 de 3383
Grupos processados: 2600 de 3383
Grupos processados: 2700 de 3383
Grupos processados: 2800 de 3383
Grupos processados: 2900 de 3383
Grupos processados: 3000 de 3383
Grupos processados:

,data_fim_janela,ativo_1,ativo_2,data_entrada,data_saida,dias_abertos,posicao,z_entrada,z_saida,beta_entrada,beta_saida,pnl_bruto,pnl_liquido,razao_saida
0,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,2012-03-16,2012-03-21,3,short_ativo_1_long_ativo_2,2.592098,0.338973,0.511731,0.465078,0.223144,0.221144,convergencia
1,2012-03-31,BRTOYBACNOR4,BRTOYBACNPR1,2012-04-20,2012-04-30,6,long_ativo_1_short_ativo_2,-2.379135,1.164552,0.454729,0.204118,0.418499,0.416499,fim_janela
2,2012-04-30,BRTOYBACNOR4,BRTOYBACNPR1,2012-05-10,2012-05-14,2,long_ativo_1_short_ativo_2,-2.404738,0.231975,0.424524,0.483832,0.287682,0.285682,convergencia
3,2012-05-31,BRTOYBACNOR4,BRTOYBACNPR1,2012-06-14,2012-06-15,1,long_ativo_1_short_ativo_2,-3.098232,0.232120,0.113758,0.123976,0.405465,0.403465,convergencia
4,2012-05-31,BRTOYBACNOR4,BRTOYBACNPR1,2012-06-18,2012-06-19,1,long_ativo_1_short_ativo_2,-2.219522,0.282194,0.238903,0.250073,0.308598,0.306598,convergencia


### Adição de informações dos pares aos trades

Nesta etapa, adicionamos aos trades algumas informações que estavam na base de sinais, como tickers, setores, p-valor da cointegração e distância.

Essas informações ajudam a analisar depois quais tipos de pares tiveram melhor performance.

In [9]:
colunas_info = [
    "data_fim_janela",
    "ativo_1",
    "ativo_2",
    "ticker_1",
    "ticker_2",
    "setor_1",
    "setor_2",
    "mesmo_setor",
    "combinacao_setorial",
    "ranking_cointegracao",
    "ranking_distancia",
    "pvalor_coint",
    "distancia"
]

colunas_info_existentes = [
    coluna for coluna in colunas_info
    if coluna in sinais.columns
]

info_pares = (
    sinais[colunas_info_existentes]
    .drop_duplicates(subset=["data_fim_janela", "ativo_1", "ativo_2"])
)

if not trades.empty:
    trades = trades.merge(
        info_pares,
        on=["data_fim_janela", "ativo_1", "ativo_2"],
        how="left"
    )

display(trades.head())

,data_fim_janela,ativo_1,ativo_2,data_entrada,data_saida,dias_abertos,posicao,z_entrada,z_saida,beta_entrada,beta_saida,pnl_bruto,pnl_liquido,razao_saida,ticker_1,ticker_2,setor_1,setor_2,mesmo_setor,combinacao_setorial,ranking_cointegracao,ranking_distancia,pvalor_coint,distancia
0,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,2012-03-16,2012-03-21,3,short_ativo_1_long_ativo_2,2.592098,0.338973,0.511731,0.465078,0.223144,0.221144,convergencia,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,1,19,0.000002,1.388557
1,2012-03-31,BRTOYBACNOR4,BRTOYBACNPR1,2012-04-20,2012-04-30,6,long_ativo_1_short_ativo_2,-2.379135,1.164552,0.454729,0.204118,0.418499,0.416499,fim_janela,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,2,17,0.000008,1.382239
2,2012-04-30,BRTOYBACNOR4,BRTOYBACNPR1,2012-05-10,2012-05-14,2,long_ativo_1_short_ativo_2,-2.404738,0.231975,0.424524,0.483832,0.287682,0.285682,convergencia,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,1,248,0.000003,3.886214
3,2012-05-31,BRTOYBACNOR4,BRTOYBACNPR1,2012-06-14,2012-06-15,1,long_ativo_1_short_ativo_2,-3.098232,0.232120,0.113758,0.123976,0.405465,0.403465,convergencia,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,2,98,0.000003,3.726570
4,2012-05-31,BRTOYBACNOR4,BRTOYBACNPR1,2012-06-18,2012-06-19,1,long_ativo_1_short_ativo_2,-2.219522,0.282194,0.238903,0.250073,0.308598,0.306598,convergencia,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,2,98,0.000003,3.726570


### Cálculo do PnL diário e da curva acumulada

Nesta etapa, transformamos os resultados dos trades em uma série diária.

O PnL de cada trade será atribuído à data de saída da operação.

Em dias sem fechamento de trade, o PnL será considerado zero.

A curva acumulada será calculada pela soma acumulada dos PnLs líquidos.

In [10]:
if trades.empty:
    raise ValueError("Nenhum trade foi gerado. Verifique os parâmetros de entrada, saída e stop.")

trades["data_entrada"] = pd.to_datetime(trades["data_entrada"])
trades["data_saida"] = pd.to_datetime(trades["data_saida"])

data_inicio = trades["data_entrada"].min()
data_fim = trades["data_saida"].max()

datas_backtest = matriz_log.loc[
    (matriz_log.index >= data_inicio) &
    (matriz_log.index <= data_fim)
].index

pnl_por_dia = (
    trades
    .groupby("data_saida")["pnl_liquido"]
    .sum()
)

pnl_diario = (
    pnl_por_dia
    .reindex(datas_backtest, fill_value=0.0)
    .rename("pnl_diario")
    .to_frame()
)

pnl_diario["pnl_acumulado"] = pnl_diario["pnl_diario"].cumsum()

pnl_diario["pico_acumulado"] = pnl_diario["pnl_acumulado"].cummax()

pnl_diario["drawdown"] = (
    pnl_diario["pnl_acumulado"] - pnl_diario["pico_acumulado"]
)

display(pnl_diario.head())

display(pnl_diario.tail())

,pnl_diario,pnl_acumulado,pico_acumulado,drawdown
data,,,,
2012-03-16,0.000000,0.000000,0.000000,0.000000
2012-03-19,0.000000,0.000000,0.000000,0.000000
2012-03-20,0.000000,0.000000,0.000000,0.000000
2012-03-21,0.221144,0.221144,0.221144,0.000000
2012-03-22,0.000000,0.221144,0.221144,0.000000


,pnl_diario,pnl_acumulado,pico_acumulado,drawdown
data,,,,
2026-03-13,0.000000,10.008638,10.087502,-0.078864
2026-03-16,0.000000,10.008638,10.087502,-0.078864
2026-03-17,0.000000,10.008638,10.087502,-0.078864
2026-03-18,0.000000,10.008638,10.087502,-0.078864
2026-03-19,0.090968,10.099606,10.099606,0.000000


### Cálculo das métricas gerais de performance

Nesta etapa, calculamos as principais métricas do backtest:

- número de trades;
- win rate;
- PnL total;
- PnL médio;
- volatilidade diária;
- Sharpe anualizado;
- máximo drawdown;
- profit factor;
- payoff ratio;
- duração média dos trades.

In [11]:
n_trades = len(trades)

trades_vencedores = trades[trades["pnl_liquido"] > 0]
trades_perdedores = trades[trades["pnl_liquido"] < 0]

win_rate = len(trades_vencedores) / n_trades if n_trades > 0 else np.nan

pnl_total = trades["pnl_liquido"].sum()

pnl_medio = trades["pnl_liquido"].mean()

vol_diaria = pnl_diario["pnl_diario"].std()

sharpe = (
    pnl_diario["pnl_diario"].mean() / vol_diaria * np.sqrt(252)
    if vol_diaria > 0
    else np.nan
)

max_drawdown = pnl_diario["drawdown"].min()

ganho_total = trades_vencedores["pnl_liquido"].sum()

perda_total = abs(trades_perdedores["pnl_liquido"].sum())

profit_factor = ganho_total / perda_total if perda_total > 0 else np.inf

ganho_medio = trades_vencedores["pnl_liquido"].mean()

perda_media = abs(trades_perdedores["pnl_liquido"].mean())

payoff_ratio = ganho_medio / perda_media if perda_media > 0 else np.inf

duracao_media = trades["dias_abertos"].mean()

metricas_gerais = pd.DataFrame([{
    "n_trades": n_trades,
    "win_rate": win_rate,
    "pnl_total": pnl_total,
    "pnl_medio": pnl_medio,
    "vol_diaria": vol_diaria,
    "sharpe_anualizado": sharpe,
    "max_drawdown": max_drawdown,
    "profit_factor": profit_factor,
    "payoff_ratio": payoff_ratio,
    "duracao_media": duracao_media
}])

display(metricas_gerais)

,n_trades,win_rate,pnl_total,pnl_medio,vol_diaria,sharpe_anualizado,max_drawdown,profit_factor,payoff_ratio,duracao_media
0,272,0.625000,10.099606,0.037131,0.037941,1.216725,-1.149320,2.630900,1.578540,3.136029


### Função de métricas por grupo

Nesta etapa, criamos uma função para calcular métricas de performance por diferentes agrupamentos.

Essa função será usada para analisar performance por:

- par;
- ano;
- setor;
- razão de saída.

In [12]:
def calcular_metricas_grupo(grupo):
    n = len(grupo)
    
    vencedores = grupo[grupo["pnl_liquido"] > 0]
    perdedores = grupo[grupo["pnl_liquido"] < 0]
    
    win_rate = len(vencedores) / n if n > 0 else np.nan
    
    pnl_total = grupo["pnl_liquido"].sum()
    
    pnl_medio = grupo["pnl_liquido"].mean()
    
    ganho_total = vencedores["pnl_liquido"].sum()
    
    perda_total = abs(perdedores["pnl_liquido"].sum())
    
    profit_factor = ganho_total / perda_total if perda_total > 0 else np.inf
    
    duracao_media = grupo["dias_abertos"].mean()
    
    return pd.Series({
        "n_trades": n,
        "win_rate": win_rate,
        "pnl_total": pnl_total,
        "pnl_medio": pnl_medio,
        "profit_factor": profit_factor,
        "duracao_media": duracao_media
    })

### Métricas por par

Nesta etapa, avaliamos quais pares tiveram melhor e pior performance no backtest.

Isso ajuda a identificar se a estratégia depende de poucos pares ou se a performance é distribuída.

In [13]:
metricas_por_par = (
    trades
    .groupby(["ativo_1", "ativo_2"])
    .apply(calcular_metricas_grupo)
    .reset_index()
    .sort_values("pnl_total", ascending=False)
)

display(metricas_por_par.head(20))

,ativo_1,ativo_2,n_trades,win_rate,pnl_total,pnl_medio,profit_factor,duracao_media
153,BRTOYBACNOR4,BRTOYBACNPR1,9.000000,0.888889,2.860454,0.317828,"1,431.226769",5.000000
1,BRADHMACNOR9,BRHOOTACNPR9,3.000000,1.000000,0.890628,0.296876,inf,3.333333
154,BRTOYBACNPR1,BRUSIMACNPA6,3.000000,1.000000,0.780934,0.260311,inf,2.666667
132,BRRNEWACNOR8,BRRNEWACNPR5,14.000000,0.857143,0.667415,0.047672,9.253548,3.357143
134,BRRNEWACNPR5,BRRNEWCDAM15,15.000000,0.866667,0.609970,0.040665,11.435159,2.000000
121,BROGXPACNOR3,BRVIVRACNOR4,1.000000,1.000000,0.523178,0.523178,inf,8.000000
118,BRMOTVACNOR7,BRVIVTACNPR7,3.000000,0.666667,0.515233,0.171744,5.630722,2.333333
49,BRCBMAACNPR1,BRDASAACNOR1,2.000000,1.000000,0.452575,0.226288,inf,3.000000
138,BRSANBACNOR8,BRSANBCDAM13,6.000000,0.833333,0.406347,0.067724,204.173288,1.500000
77,BRENGICDAM16,BRLRENACNOR1,2.000000,1.000000,0.333746,0.166873,inf,5.000000


### Métricas por ano

Nesta etapa, analisamos a performance da estratégia por ano.

Isso permite verificar se o resultado foi consistente ao longo do tempo ou concentrado em poucos períodos.

In [14]:
trades["ano_saida"] = trades["data_saida"].dt.year

metricas_por_ano = (
    trades
    .groupby("ano_saida")
    .apply(calcular_metricas_grupo)
    .reset_index()
    .sort_values("ano_saida")
)

display(metricas_por_ano)

,ano_saida,n_trades,win_rate,pnl_total,pnl_medio,profit_factor,duracao_media
0,2012,17.000000,0.941176,3.859188,0.227011,"1,930.593918",3.941176
1,2013,21.000000,0.619048,1.052267,0.050108,5.169331,4.000000
2,2014,23.000000,0.608696,0.687593,0.029895,4.143091,2.956522
3,2015,22.000000,0.454545,0.279101,0.012686,1.345391,4.090909
4,2016,15.000000,0.600000,0.727588,0.048506,3.840414,4.133333
5,2017,10.000000,0.600000,-0.100176,-0.010018,0.827327,3.200000
6,2018,19.000000,0.368421,-0.057035,-0.003002,0.905619,2.368421
7,2019,20.000000,0.500000,-0.584503,-0.029225,0.484106,2.550000
8,2020,34.000000,0.735294,2.861791,0.084170,6.804170,2.705882
9,2021,11.000000,0.363636,-0.206645,-0.018786,0.528324,2.909091


### Métricas por setor e combinação setorial

Nesta etapa, analisamos a performance por setor e por combinação setorial.

Essa análise é importante porque a formação dos pares não foi restrita por setor.

Assim, podemos avaliar posteriormente se pares do mesmo setor, de setores diferentes ou de determinadas combinações setoriais tiveram desempenho superior.

In [15]:
if "setor_1" in trades.columns:
    metricas_por_setor = (
        trades
        .dropna(subset=["setor_1"])
        .groupby("setor_1")
        .apply(calcular_metricas_grupo)
        .reset_index()
        .sort_values("pnl_total", ascending=False)
    )
    
    display(metricas_por_setor)
else:
    metricas_por_setor = pd.DataFrame()
    print("Coluna setor_1 não encontrada.")

if "combinacao_setorial" in trades.columns:
    metricas_por_combinacao = (
        trades
        .dropna(subset=["combinacao_setorial"])
        .groupby("combinacao_setorial")
        .apply(calcular_metricas_grupo)
        .reset_index()
        .sort_values("pnl_total", ascending=False)
    )
    
    display(metricas_por_combinacao.head(20))
else:
    metricas_por_combinacao = pd.DataFrame()
    print("Coluna combinacao_setorial não encontrada.")

,setor_1,n_trades,win_rate,pnl_total,pnl_medio,profit_factor,duracao_media
3,Consumo cíclico,48.000000,0.541667,3.577044,0.074522,3.210798,2.854167
11,Utilidade pública,77.000000,0.792208,2.297192,0.029834,4.565899,2.909091
5,Financeiro,64.000000,0.531250,1.336324,0.020880,1.931845,3.140625
1,Bens industriais,38.000000,0.526316,1.102840,0.029022,1.746023,3.447368
9,Saúde,7.000000,0.571429,0.994316,0.142045,36.023296,3.142857
8,Petróleo gás e biocombustíveis,6.000000,0.666667,0.760746,0.126791,8.150687,6.000000
6,Materiais básicos,13.000000,0.923077,0.338500,0.026038,170.250047,1.307692
7,Outros,1.000000,1.000000,0.138656,0.138656,inf,5.000000
0,-,3.000000,1.000000,0.090142,0.030047,inf,6.666667
10,Tecnologia da informação,1.000000,1.000000,0.042850,0.042850,inf,9.000000


,combinacao_setorial,n_trades,win_rate,pnl_total,pnl_medio,profit_factor,duracao_media
14,Consumo cíclico | Consumo cíclico,22.000000,0.681818,3.090663,0.140485,10.152923,3.727273
51,Utilidade pública | Utilidade pública,58.000000,0.810345,1.626947,0.028051,4.671314,2.551724
41,Saúde | Consumo cíclico,5.000000,0.800000,1.020706,0.204141,511.353163,2.600000
29,Financeiro | Financeiro,26.000000,0.615385,0.790252,0.030394,2.821246,3.115385
39,Petróleo gás e biocombustíveis | Consumo cíclico,6.000000,0.666667,0.760746,0.126791,8.150687,6.000000
17,Consumo cíclico | Materiais básicos,4.000000,0.750000,0.730465,0.182616,15.473705,3.750000
27,Financeiro | Consumo cíclico,9.000000,0.666667,0.538044,0.059783,31.239739,3.777778
4,Bens industriais | Comunicações,3.000000,0.666667,0.515233,0.171744,5.630722,2.333333
8,Bens industriais | Saúde,2.000000,1.000000,0.452575,0.226288,inf,3.000000
46,Utilidade pública | Consumo cíclico,5.000000,0.600000,0.338736,0.067747,50.700922,3.200000


### Métricas por razão de saída

Nesta etapa, analisamos por que os trades foram encerrados.

As saídas podem ocorrer por:

- convergência;
- stop por divergência;
- stop por tempo;
- fim da janela.

Essa análise mostra se a estratégia está fechando mais operações por convergência ou por stop.

In [16]:
metricas_por_saida = (
    trades
    .groupby("razao_saida")
    .apply(calcular_metricas_grupo)
    .reset_index()
    .sort_values("n_trades", ascending=False)
)

display(metricas_por_saida)

,razao_saida,n_trades,win_rate,pnl_total,pnl_medio,profit_factor,duracao_media
0,convergencia,143.000000,0.909091,11.630470,0.081332,30.316853,3.328671
1,fim_janela,89.000000,0.325843,0.954542,0.010725,1.501881,3.471910
2,stop_divergencia,40.000000,0.275000,-2.485407,-0.062135,0.361737,1.700000


### Salvamento dos resultados do backtest

Nesta etapa, salvamos as principais saídas do backtest:

- base de trades;
- métricas gerais;
- PnL diário;
- métricas por par;
- métricas por ano;
- métricas por setor;
- métricas por combinação setorial;
- métricas por razão de saída.

In [17]:
trades.to_parquet(arquivo_trades, index=False)

metricas_gerais.to_parquet(arquivo_metricas, index=False)

pnl_diario.reset_index(names="data").to_parquet(arquivo_pnl_diario, index=False)

metricas_por_par.to_parquet("../dados_tratados/backtest_metricas_por_par.parquet", index=False)

metricas_por_ano.to_parquet("../dados_tratados/backtest_metricas_por_ano.parquet", index=False)

metricas_por_saida.to_parquet("../dados_tratados/backtest_metricas_por_saida.parquet", index=False)

if not metricas_por_setor.empty:
    metricas_por_setor.to_parquet("../dados_tratados/backtest_metricas_por_setor.parquet", index=False)

if not metricas_por_combinacao.empty:
    metricas_por_combinacao.to_parquet("../dados_tratados/backtest_metricas_por_combinacao.parquet", index=False)

print("Backtest salvo com sucesso.")

Backtest salvo com sucesso.
